In [20]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

In [1]:
import pandas as pd

# 입력 파일 경로
input_path = r"C:\Users\Admin\OneDrive\바탕 화면\인사평가ver2\인사평가_환율적용.csv"
output_path = r"C:\Users\Admin\OneDrive\바탕 화면\인사평가ver2\인사평가_환율적용_제외열저장.csv"

# 제외할 열
exclude_cols = ["나이", "회사의 마찰", "학력", "사번", "만족도", "성별", "야근"]

# CSV 불러오기
df = pd.read_csv(input_path)

# 제외할 열이 실제로 존재하는 것만 제거
cols_to_drop = [col for col in exclude_cols if col in df.columns]
df_filtered = df.drop(columns=cols_to_drop)

# 저장
df_filtered.to_csv(output_path, index=False)

output_path


'C:\\Users\\Admin\\OneDrive\\바탕 화면\\인사평가ver2\\인사평가_환율적용_제외열저장.csv'

In [5]:
!pip install tensorflow==2.16.1


  Using cached protobuf-4.25.8-cp310-abi3-win_amd64.whl.metadata (541 bytes)
   ---------------------------------------- 0.0/376.9 MB ? eta -:--:--
   ---------------------------------------- 1.3/376.9 MB 11.3 MB/s eta 0:00:34
   ---------------------------------------- 3.9/376.9 MB 11.8 MB/s eta 0:00:32
    --------------------------------------- 6.6/376.9 MB 11.8 MB/s eta 0:00:32
    --------------------------------------- 8.4/376.9 MB 10.8 MB/s eta 0:00:35
   - -------------------------------------- 10.2/376.9 MB 10.5 MB/s eta 0:00:36
   - -------------------------------------- 12.8/376.9 MB 10.7 MB/s eta 0:00:34
   - -------------------------------------- 15.2/376.9 MB 10.9 MB/s eta 0:00:34
   - -------------------------------------- 17.3/376.9 MB 10.7 MB/s eta 0:00:34
   -- ------------------------------------- 19.7/376.9 MB 10.7 MB/s eta 0:00:34
   -- ------------------------------------- 21.8/376.9 MB 10.7 MB/s eta 0:00:34
   -- ------------------------------------- 23.6/376.9 M

In [6]:
# -*- coding: utf-8 -*-
import warnings, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier

# ===== 한글 폰트(윈도우) =====
plt.rc("font", family="Malgun Gothic")
plt.rc("axes", unicode_minus=False)

# XGBoost
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# DNN (TensorFlow Keras) -> 없으면 MLPClassifier로 대체
USE_KERAS = True
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
except Exception:
    USE_KERAS = False
    from sklearn.neural_network import MLPClassifier

# ===== 경로 =====
INPUT_CSV = r"C:\Users\Admin\OneDrive\바탕 화면\인사평가ver2\인사평가_환율적용_제외열저장.csv"
OUTDIR    = Path("./reports"); OUTDIR.mkdir(parents=True, exist_ok=True)

# ===== 유틸: 텍스트 리포트를 이미지로 저장 =====
def save_text_report_as_png(report_text: str, model_name: str, outdir: Path = OUTDIR):
    fig = plt.figure(figsize=(7.5, 3.8), dpi=150)
    ax = plt.gca()
    ax.axis("off")
    y = 1.0; line_h = 0.055
    for line in report_text.splitlines():
        ax.text(0.02, y, line, family="monospace", fontsize=10, va="top")
        y -= line_h
    fig.tight_layout()
    out_path = outdir / f"report_{model_name}.png"
    plt.savefig(out_path, bbox_inches="tight", pad_inches=0.2)
    plt.close(fig)
    return out_path

# ===== 유틸: 혼동행렬 저장 =====
def save_confusion_matrix(y_true, y_pred, model_name: str, outdir: Path = OUTDIR):
    labels = ["보통", "좋다"]
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(4.8, 4.2), dpi=150)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"Confusion Matrix - {model_name}")
    fig.tight_layout()
    out_path = outdir / f"cm_{model_name}.png"
    plt.savefig(out_path, bbox_inches="tight", pad_inches=0.2)
    plt.close(fig)
    return out_path

# ===== 데이터 로드 =====
df = pd.read_csv(INPUT_CSV, encoding="cp949")   # 필요시 "euc-kr"

# ===== 타깃 / 피처 =====
if "업무평가" not in df.columns:
    raise ValueError("데이터에 '업무평가' 열이 없습니다.")
y_raw = df["업무평가"]
X = df.drop(columns=["업무평가"])

# 타깃 인코딩(보통=0, 그 외=1)
if y_raw.dtype == "object":
    y = (y_raw != "보통").astype(int)
else:
    uniq = set(pd.unique(y_raw))
    y = y_raw.astype(int) if uniq <= {0,1} else (y_raw != 0).astype(int)

# ===== 열 타입 분리 =====
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

# ===== 전처리 파이프라인 =====
num_tf = Pipeline([("imp", SimpleImputer(strategy="median"))])
cat_tf = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])
prep = ColumnTransformer([("num", num_tf, num_cols),
                          ("cat", cat_tf, cat_cols)])

# ===== 데이터 분할 =====
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ===== 공통: 학습/평가/시각화 저장 =====
def fit_eval_report(model, name: str):
    pipe = Pipeline([("prep", prep), ("clf", model)])
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    acc = accuracy_score(y_te, pred)
    report = classification_report(y_te, pred, target_names=["보통","좋다"], digits=2)
    header = f"[Classification Report]  ({name})\naccuracy: {acc:.2f}\n"
    final_text = header + report
    print(final_text)
    png_rep = save_text_report_as_png(final_text, name)
    png_cm  = save_confusion_matrix(y_te, pred, name)
    print(f"→ 저장: {png_rep}")
    print(f"→ 저장: {png_cm}\n")
    return pipe

# ====== 1) DNN ======
if USE_KERAS:
    class KerasClassifierWrapper:
        def __init__(self, epochs=60, batch_size=32, patience=8, verbose=0):
            self.epochs = epochs; self.batch_size = batch_size
            self.patience = patience; self.verbose = verbose
            self.model_ = None; self.prep_ = None
        def fit(self, X, y):
            self.prep_ = prep
            Xp = self.prep_.fit_transform(X, y)
            in_dim = Xp.shape[1]
            self.model_ = keras.Sequential([
                layers.Input(shape=(in_dim,)),
                layers.Dense(128, activation="relu"),
                layers.Dense(64, activation="relu"),
                layers.Dense(1, activation="sigmoid")
            ])
            self.model_.compile(optimizer=keras.optimizers.Adam(1e-3),
                                loss="binary_crossentropy",
                                metrics=["accuracy"])
            cb = [keras.callbacks.EarlyStopping(monitor="val_loss",
                                                patience=self.patience,
                                                restore_best_weights=True)]
            self.model_.fit(Xp, y, epochs=self.epochs, batch_size=self.batch_size,
                            validation_split=0.2, callbacks=cb, verbose=self.verbose)
            return self
        def predict(self, X):
            Xp = self.prep_.transform(X)
            prob = self.model_.predict(Xp, verbose=0).ravel()
            return (prob >= 0.5).astype(int)

    print("=== DNN(Keras) 학습 ===")
    dnn = KerasClassifierWrapper()
    dnn.prep_ = prep
    dnn.fit(X_tr, y_tr)
    dnn_pred = dnn.predict(X_te)
    dnn_acc = accuracy_score(y_te, dnn_pred)
    dnn_report = classification_report(y_te, dnn_pred, target_names=["보통","좋다"], digits=2)
    dnn_text = f"[Classification Report]  (DNN)\naccuracy: {dnn_acc:.2f}\n" + dnn_report
    print(dnn_text)
    save_text_report_as_png(dnn_text, "DNN")
    save_confusion_matrix(y_te, dnn_pred, "DNN")
else:
    print("TensorFlow가 없어 MLPClassifier로 대체합니다.")
    mlp = MLPClassifier(hidden_layer_sizes=(128,64), activation="relu",
                        solver="adam", max_iter=300, random_state=42)
    fit_eval_report(mlp, "MLP")

# ====== 2) RandomForest ======
rf = RandomForestClassifier(
    n_estimators=500, max_depth=None,
    min_samples_split=2, min_samples_leaf=1,
    n_jobs=-1, random_state=42
)
fit_eval_report(rf, "RandomForest")

# ====== 3) XGBoost ======
if HAS_XGB:
    xgb = XGBClassifier(
        n_estimators=600, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        objective="binary:logistic", tree_method="hist",
        eval_metric="logloss", random_state=42
    )
    fit_eval_report(xgb, "XGBoost")
else:
    print("XGBoost 미설치: 건너뜁니다.")


TensorFlow가 없어 MLPClassifier로 대체합니다.
[Classification Report]  (MLP)
accuracy: 0.85
              precision    recall  f1-score   support

          보통       0.85      1.00      0.92       249
          좋다       0.00      0.00      0.00        45

    accuracy                           0.85       294
   macro avg       0.42      0.50      0.46       294
weighted avg       0.72      0.85      0.78       294

→ 저장: reports\report_MLP.png
→ 저장: reports\cm_MLP.png

[Classification Report]  (RandomForest)
accuracy: 0.85
              precision    recall  f1-score   support

          보통       0.85      1.00      0.92       249
          좋다       0.00      0.00      0.00        45

    accuracy                           0.85       294
   macro avg       0.42      0.50      0.46       294
weighted avg       0.72      0.85      0.78       294

→ 저장: reports\report_RandomForest.png
→ 저장: reports\cm_RandomForest.png

[Classification Report]  (XGBoost)
accuracy: 0.83
              precision    rec

In [19]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
import matplotlib as mpl
import os

# ===== 0) 한글 폰트 =====
try:
    mpl.rcParams["font.family"] = "Malgun Gothic"   # Windows
except Exception:
    pass
plt.rcParams["axes.unicode_minus"] = False

# ===== 1) 데이터 로드 =====
CSV_PATH = r"C:\Users\Admin\OneDrive\바탕 화면\인사평가ver2\인사평가_환율적용_제외열저장.csv"
df = pd.read_csv(CSV_PATH, encoding="cp949")

# ===== 2) 타깃/피처 =====
target_col = "업무평가"
y = df[target_col]
X = df.drop(columns=[target_col])

# (선택) 특정 열 제외
# drop_cols = ["성인여부", "근무기준시간", "결혼여부"]
# X = X.drop(columns=[c for c in drop_cols if c in X.columns])

# ===== 3) 범주/수치 구분 =====
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

# ===== 4) 전처리 + 모델 =====
cat_tf = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])
num_tf = Pipeline([("imp", SimpleImputer(strategy="median"))])

prep = ColumnTransformer([
    ("num", num_tf, num_cols),
    ("cat", cat_tf, cat_cols),
])

xgb = XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, use_label_encoder=False, eval_metric="logloss"
)

pipe = Pipeline([("prep", prep), ("clf", xgb)])

# ===== 5) 학습 =====
y_enc, _ = pd.factorize(y)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)
pipe.fit(X_tr, y_tr)

# ===== 6) 전처리 후 특성명 & 중요도 =====
feat_names = pipe.named_steps["prep"].get_feature_names_out()
importances = pipe.named_steps["clf"].feature_importances_

# prefix 제거
def strip_prefix(n: str) -> str:
    return n.replace("num__", "").replace("cat__", "")
clean_names = [strip_prefix(n) for n in feat_names]

fi = pd.DataFrame({"feature": clean_names, "importance": importances})
fi = fi.sort_values("importance", ascending=False, ignore_index=True)

# ===== 7) 누적 중요도(cumsum) =====
fi["cumsum"] = fi["importance"].cumsum()
fi["rank"] = np.arange(1, len(fi) + 1)

# ===== 8) 그림 (누적선만) =====
plt.figure(figsize=(12, 8))
plt.plot(fi["rank"], fi["cumsum"], marker="o", linewidth=2)
plt.ylim(0, 1.02)
plt.xlim(1, len(fi))
plt.title("Cumulative Feature Importance (XGBoost)")
plt.ylabel("Cumulative Importance")

# x축을 피처명으로 표시
plt.xticks(fi["rank"], fi["feature"], rotation=60, ha="right")
plt.grid(alpha=0.3)

# ===== 9) 저장 =====
out_dir = os.path.dirname(CSV_PATH)
out_path = os.path.join(out_dir, "feature_importance_cumsum.png")
plt.tight_layout()
plt.savefig(out_path, dpi=220, bbox_inches="tight")
plt.close()

print("저장:", out_path)


저장: C:\Users\Admin\OneDrive\바탕 화면\인사평가ver2\feature_importance_cumsum.png


In [22]:
# -*- coding: utf-8 -*-
"""
인사평가 - XGBoost / RandomForest / DNN(없으면 MLP) 간단 평가 출력
- 결과는 accuracy와 classification_report만 콘솔로 출력
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

RANDOM_STATE = 42
INPUT_CSV = r"C:\Users\Admin\OneDrive\바탕 화면\2차 플젝\인사평가_피쳐생성.csv"
TARGET_COL = "업무평가"
TEST_SIZE = 0.2

# ===== 데이터 로드 =====
encs = ["utf-8-sig", "cp949", "euc-kr"]
for enc in encs:
    try:
        df = pd.read_csv(INPUT_CSV, encoding=enc)
        break
    except Exception:
        if enc == encs[-1]:
            raise

if TARGET_COL not in df.columns:
    raise ValueError(f"타깃 열 '{TARGET_COL}' 이(가) 없습니다.")

y_raw = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL])

# 범주/수치
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

# 라벨 인코딩(문자 타깃 대응)
le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = [str(c) for c in le.classes_]
n_classes = len(class_names)
is_binary = (n_classes == 2)

# ===== 데이터 분할 =====
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

# ===== 전처리 =====
num_tf = Pipeline([("imp", SimpleImputer(strategy="median"))])
cat_tf = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

prep = ColumnTransformer([
    ("num", num_tf, num_cols),
    ("cat", cat_tf, cat_cols),
], remainder="drop")

def evaluate_model(name, pipe):
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    print(f"\n[{name}] accuracy: {acc:.2f}")
    print(classification_report(y_te, y_pred, target_names=class_names, zero_division=0))

# ===== 1) RandomForest =====
rf = Pipeline([
    ("prep", prep),
    ("clf", RandomForestClassifier(
        n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1
    ))
])
evaluate_model("RandomForest", rf)

# ===== 2) XGBoost =====
xgb_params = dict(
    n_estimators=600, max_depth=5, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
    tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE
)
if is_binary:
    xgb_params.update(objective="binary:logistic", eval_metric="logloss")
else:
    xgb_params.update(objective="multi:softprob", num_class=n_classes, eval_metric="mlogloss")

xgb = Pipeline([
    ("prep", prep),
    ("clf", XGBClassifier(**xgb_params))
])
evaluate_model("XGBoost", xgb)

# ===== 3) DNN (없으면 MLP로 대체) =====
USE_DNN = True
try:
    import tensorflow as tf
    from tensorflow import keras
except Exception:
    USE_DNN = False

if USE_DNN:
    # 전처리 후 배열로 변환
    prep.fit(X_tr)
    Xtr = prep.transform(X_tr)
    Xte = prep.transform(X_te)

    input_dim = Xtr.shape[1]
    if is_binary:
        model = keras.Sequential([
            keras.layers.Input(shape=(input_dim,)),
            keras.layers.Dense(128, activation="relu"),
            keras.layers.Dropout(0.2),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dropout(0.2),
            keras.layers.Dense(1, activation="sigmoid"),
        ])
        model.compile(optimizer=keras.optimizers.Adam(1e-3),
                      loss="binary_crossentropy",
                      metrics=["accuracy"])
    else:
        model = keras.Sequential([
            keras.layers.Input(shape=(input_dim,)),
            keras.layers.Dense(192, activation="relu"),
            keras.layers.Dropout(0.3),
            keras.layers.Dense(96, activation="relu"),
            keras.layers.Dropout(0.3),
            keras.layers.Dense(n_classes, activation="softmax"),
        ])
        model.compile(optimizer=keras.optimizers.Adam(1e-3),
                      loss=keras.losses.SparseCategoricalCrossentropy(),
                      metrics=["accuracy"])

    cb = [keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True)]
    model.fit(Xtr, y_tr, validation_split=0.2, epochs=150, batch_size=64, callbacks=cb, verbose=0)

    # 예측 -> 리포트
    if is_binary:
        ypred = (model.predict(Xte, verbose=0).ravel() >= 0.5).astype(int)
    else:
        ypred = model.predict(Xte, verbose=0).argmax(1)

    acc = accuracy_score(y_te, ypred)
    print(f"\n[DNN] accuracy: {acc:.2f}")
    print(classification_report(y_te, ypred, target_names=class_names, zero_division=0))

else:
    # ===== 대체: MLPClassifier =====
    mlp = Pipeline([
        ("prep", prep),
        ("clf", MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            solver="adam",
            batch_size=64,
            learning_rate_init=1e-3,
            max_iter=200,
            random_state=RANDOM_STATE
        ))
    ])
    evaluate_model("MLP(fallback)", mlp)



[RandomForest] accuracy: 0.85
              precision    recall  f1-score   support

          보통       0.85      1.00      0.92       249
          좋다       0.00      0.00      0.00        45

    accuracy                           0.85       294
   macro avg       0.42      0.50      0.46       294
weighted avg       0.72      0.85      0.78       294


[XGBoost] accuracy: 0.84
              precision    recall  f1-score   support

          보통       0.85      0.98      0.91       249
          좋다       0.33      0.04      0.08        45

    accuracy                           0.84       294
   macro avg       0.59      0.51      0.50       294
weighted avg       0.77      0.84      0.78       294


[MLP(fallback)] accuracy: 0.85
              precision    recall  f1-score   support

          보통       0.85      1.00      0.92       249
          좋다       0.00      0.00      0.00        45

    accuracy                           0.85       294
   macro avg       0.42      0.50      